In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/saloncut_data"
TEST_NORMAL = f"{BASE}/test_images/normal"
TEST_FAIL = f"{BASE}/test_images/fail"

import os
def list_images(path):
    return sorted(f for f in os.listdir(path) if f.lower().endswith(('.jpg', '.png')))

print("정상:", list_images(TEST_NORMAL))
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!rm -rf /content/SalonCutAI
!git clone -q -b dev https://github.com/qja0707/SalonCutAI.git

import sys
sys.path.append("/content/SalonCutAI/backend/src/ai_engine")

from image_gen.base import load_pipeline, generate, release
print("import 성공")

# 조합 2 — InstantID + SDXL

얼굴 임베딩과 IdentityNet(ControlNet)으로 참조 얼굴을 재현하는 방식.
이미지 전체를 재생성하므로 헤어 변형 위험이 있다.

## 조합 1 제외 기록

조합 1(inswapper)은 공식 배포 저장소가 접근 불가(401) 상태로 전환되어 제외한다.
비공식 미러는 원본 동일성 검증이 불가하고 배포 중단 의도에 반하므로 사용하지 않는다.
계획서의 모델 착수 확인 절차 1번(현재 배포 중인가)에 해당하는 사례.

In [ ]:
!pip install -q insightface onnxruntime-gpu

# inswapper 가중치 확보 시도
from huggingface_hub import hf_hub_download
try:
    p = hf_hub_download("deepinsight/inswapper", "inswapper_128.onnx")
    print("다운로드 성공:", p)
except Exception as e:
    print("실패:", type(e).__name__, e)

In [ ]:
# InstantID는 얼굴 인코더로 antelopev2를 사용
from insightface.app import FaceAnalysis

try:
    app = FaceAnalysis(name='antelopev2', root='/content',
                       providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
    app.prepare(ctx_id=0, det_size=(640, 640))
    print("antelopev2 준비 완료")
except Exception as e:
    print("실패:", type(e).__name__, e)

In [ ]:
!find /content/models -name "*.onnx" | head -20
!echo "---"
!ls -R /content/models/antelopev2 | head -20

In [ ]:
# 압축 해제 시 경로가 한 단계 깊게 잡혀 파일을 상위로 이동
!mv /content/models/antelopev2/antelopev2/*.onnx /content/models/antelopev2/
!rmdir /content/models/antelopev2/antelopev2
!ls /content/models/antelopev2

app = FaceAnalysis(name='antelopev2', root='/content',
                   providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
app.prepare(ctx_id=0, det_size=(640, 640))
print("antelopev2 준비 완료")

In [ ]:
# CPU 버전 onnxruntime이 남아 있으면 GPU 프로바이더가 잡히지 않음
!pip uninstall -y onnxruntime onnxruntime-gpu -q
!pip install -q onnxruntime-gpu

In [ ]:
import onnxruntime as ort
print("버전:", ort.__version__)
print("사용 가능:", ort.get_available_providers())

In [ ]:
from insightface.app import FaceAnalysis

app = FaceAnalysis(name='antelopev2', root='/content',
                   providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
app.prepare(ctx_id=0, det_size=(640, 640))
print("antelopev2 준비 완료")

### 얼굴 인코더 GPU 이슈 — CPU로 진행

`onnxruntime.get_available_providers()`에는 CUDAExecutionProvider가 포함되지만
실제 세션 생성 시 CPU로 폴백한다. Colab CUDA 환경과 onnxruntime-gpu 요구 버전
불일치로 추정되며, 원인 추적 비용 대비 실익이 적어 CPU로 진행한다.

얼굴 임베딩 추출은 장당 1초 내외로 전체 생성 시간의 5% 미만이다.
팀 VM은 환경을 직접 구성하므로 성능 측정 시 재확인한다.

### InstantID 체크포인트

- IdentityNet(ControlNet) 약 2.5GB — 얼굴 위치·각도 제어
- IP-Adapter — 얼굴 임베딩 주입
- 파이프라인 코드는 저장소에 별도 `.py` 파일로 제공

In [ ]:
from huggingface_hub import hf_hub_download

try:
    # IdentityNet(ControlNet) — 얼굴 위치·각도 제어, 약 2.5GB
    hf_hub_download("InstantX/InstantID", "ControlNetModel/config.json",
                    local_dir="/content/instantid")
    hf_hub_download("InstantX/InstantID", "ControlNetModel/diffusion_pytorch_model.safetensors",
                    local_dir="/content/instantid")
    # IP-Adapter — 얼굴 임베딩 주입
    hf_hub_download("InstantX/InstantID", "ip-adapter.bin",
                    local_dir="/content/instantid")
    print("InstantID 체크포인트 확보 완료")
except Exception as e:
    print("실패:", type(e).__name__, e)

# 파이프라인 코드는 저장소에 별도 파일로 제공됨
!wget -q -O /content/pipeline_stable_diffusion_xl_instantid.py \
    https://raw.githubusercontent.com/instantX-research/InstantID/main/pipeline_stable_diffusion_xl_instantid.py
!ls -lh /content/instantid /content/pipeline_stable_diffusion_xl_instantid.py

### 파이프라인 로딩

파이프라인 코드가 저장소 내부 `ip_adapter` 모듈을 참조하므로 파일 단독으로는
import되지 않는다. 저장소 전체를 clone해야 한다.

조합 5는 모델 파일 2개(225KB·3.6MB)와 SDXL 인페인팅만 필요했던 반면,
조합 2는 체크포인트 4.1GB에 저장소 clone까지 요구한다. 구성 비용 차이가 크다.

In [ ]:
# InstantID 저장소 전체를 clone (ip_adapter 모듈 포함)
!rm -rf /content/InstantID
!git clone -q https://github.com/instantX-research/InstantID.git /content/InstantID
!ls /content/InstantID

import sys
sys.path.insert(0, "/content/InstantID")

In [ ]:
import torch, time
from diffusers.models import ControlNetModel
from pipeline_stable_diffusion_xl_instantid import StableDiffusionXLInstantIDPipeline

torch.cuda.reset_peak_memory_stats()
t0 = time.time()

# IdentityNet — 얼굴 위치·각도를 제어하는 ControlNet
controlnet = ControlNetModel.from_pretrained(
    "/content/instantid/ControlNetModel",
    torch_dtype=torch.float16,
)

# SDXL base에 IdentityNet 결합
pipe = StableDiffusionXLInstantIDPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    controlnet=controlnet,
    torch_dtype=torch.float16,
).to("cuda")

# IP-Adapter로 얼굴 임베딩 주입 경로 연결
pipe.load_ip_adapter_instantid("/content/instantid/ip-adapter.bin")

print(f"로딩 {round(time.time()-t0,1)}초 / VRAM {round(torch.cuda.max_memory_allocated()/1024**3,2)}GB")

### 참조 얼굴 생성

InstantID는 신원을 옮기는 방식이라 "어떤 얼굴로 바꿀지" 참조 이미지가 필요하다.
손님 사진을 참조로 쓰면 원본 얼굴이 그대로 재현되어 초상권 회피가 되지 않으므로,
실존하지 않는 가상 얼굴을 SDXL로 생성해 참조로 사용한다.

여성·남성 각 1장을 만들어 테스트 세트 성별에 맞춰 적용한다.

In [ ]:
import matplotlib.pyplot as plt
from image_gen.base import load_pipeline, generate, release

ref_pipe, meta = load_pipeline("stabilityai/stable-diffusion-xl-base-1.0")
print("로딩:", meta)

ref_prompts = {
    "ref_woman": ("portrait of a beautiful korean woman, clear skin, "
                  "well-groomed, pleasant expression, front facing, "
                  "studio lighting, plain background, photorealistic"),
    "ref_man":   ("portrait of a handsome korean man, clear skin, "
                  "well-groomed, pleasant expression, front facing, "
                  "studio lighting, plain background, photorealistic"),
}
ref_negative = ("blurry, low quality, deformed face, asymmetric face, "
                "distorted features, cartoon, watermark, text")

refs = {}
for name, p in ref_prompts.items():
    img, m = generate(ref_pipe, p, seed=42, negative=ref_negative,
                      steps=30, guidance=7.0, width=1024, height=1024)
    img.save(f"/content/{name}.png")
    refs[name] = img
    print(f"{name}: {m['gen_sec']}초")

# 참조 얼굴 확인 후 SDXL 해제 (InstantID 파이프라인은 유지)
del ref_pipe
print("해제 후:", release(), "GB")

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, (k, v) in zip(axes, refs.items()):
    ax.imshow(v); ax.set_title(k); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
import matplotlib.pyplot as plt
from image_gen.base import load_pipeline, generate, release

ref_pipe, meta = load_pipeline("stabilityai/stable-diffusion-xl-base-1.0")
print("로딩:", meta)

ref_prompts = {
    "ref_woman": ("portrait of a beautiful korean woman, clear skin, "
                  "well-groomed, pleasant expression, front facing, "
                  "studio lighting, plain background, photorealistic"),
    "ref_man":   ("portrait of a handsome korean man, clear skin, "
                  "well-groomed, pleasant expression, front facing, "
                  "studio lighting, plain background, photorealistic"),
}
ref_negative = ("blurry, low quality, deformed face, asymmetric face, "
                "distorted features, cartoon, watermark, text")

os.makedirs(f"{BASE}/ref_faces", exist_ok=True)      # 추가

refs = {}
for name, p in ref_prompts.items():
    img, m = generate(ref_pipe, p, seed=42, negative=ref_negative,
                      steps=30, guidance=7.0, width=1024, height=1024)
    img.save(f"{BASE}/ref_faces/{name}.png")          # 경로 변경
    refs[name] = img
    print(f"{name}: {m['gen_sec']}초")

del ref_pipe
print("해제 후:", release(), "GB")

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, (k, v) in zip(axes, refs.items()):
    ax.imshow(v); ax.set_title(k); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
import diffusers, transformers, huggingface_hub, torch
print("diffusers:", diffusers.__version__)
print("transformers:", transformers.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
print("torch:", torch.__version__)

### InstantID 생성

참조 얼굴에서 신원 임베딩을, 원본 사진에서 얼굴 위치·각도(keypoint)를 추출해
두 조건으로 이미지를 생성한다. 조합 5와 달리 이미지 전체를 새로 그리므로
헤어·의상·배경이 변형될 수 있다. 이 지점이 조합 2의 핵심 검증 대상이다.

먼저 1장으로 동작을 확인한 뒤 전체 생성으로 넘어간다.

In [ ]:
import cv2, numpy as np
from PIL import Image
from pipeline_stable_diffusion_xl_instantid import draw_kps

def prep_instantid(ref_path, src_path):
    """참조 얼굴에서 임베딩, 원본에서 얼굴 위치를 추출."""
    ref = cv2.imread(ref_path)
    src = cv2.imread(src_path)

    # 참조 얼굴 → 신원 임베딩 (어떤 얼굴로 바꿀지)
    ref_faces = app.get(ref)
    if not ref_faces:
        raise ValueError("참조 얼굴 검출 실패")
    emb = ref_faces[0]['embedding']

    # 원본 → 얼굴 5개 키포인트 (어디에 어떤 각도로 넣을지)
    src_faces = app.get(src)
    if not src_faces:
        raise ValueError("원본 얼굴 검출 실패")
    face = sorted(src_faces, key=lambda x: x['bbox'][2] * x['bbox'][3])[-1]

    src_pil = Image.open(src_path).convert("RGB")
    kps_img = draw_kps(src_pil, face['kps'])
    return emb, kps_img, src_pil


emb, kps_img, src_pil = prep_instantid(
    "/content/ref_woman.png",
    f"{TEST_NORMAL}/normal_01_short_dark.jpg",
)

# 긴 변 1024, 8의 배수 정렬 (조합 5와 동일 조건)
w, h = src_pil.size
scale = 1024 / max(w, h)
nw, nh = (int(w * scale) // 8) * 8, (int(h * scale) // 8) * 8

import time
torch.cuda.reset_peak_memory_stats()
t0 = time.time()

out = pipe(
    prompt="a person in a hair salon, natural lighting, photorealistic",
    negative_prompt="blurry, low quality, deformed, watermark, text",
    image_embeds=emb,
    image=kps_img.resize((nw, nh)),
    controlnet_conditioning_scale=0.8,   # 얼굴 위치·각도 반영 강도
    ip_adapter_scale=0.8,                # 참조 얼굴 반영 강도
    num_inference_steps=30,
    guidance_scale=5.0,
    generator=torch.Generator("cuda").manual_seed(42),
).images[0]

print(f"생성 {round(time.time()-t0,1)}초 / VRAM {round(torch.cuda.max_memory_allocated()/1024**3,2)}GB")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, im, t in zip(axes, [src_pil, kps_img, out], ["original", "keypoints", "result"]):
    ax.imshow(im); ax.set_title(t); ax.axis("off")
plt.tight_layout(); plt.show()

### InstantID img2img 재시도

text2img 방식은 원본에서 얼굴 키포인트 5개만 조건으로 받아 나머지를 모두 창작한다.
결과적으로 헤어·의상·배경이 전부 변형되어 MVP 요구("얼굴만 교체")를 만족하지 못했다.

InstantID가 공식 제공하는 img2img 파이프라인은 원본 이미지 자체를 입력으로 받아
strength만큼만 변형한다. 원본 구도가 유지되는지 확인한다.

In [ ]:
# 기존 text2img 파이프라인 해제 후 img2img로 교체
del pipe
print("해제 후:", release(), "GB")

from pipeline_stable_diffusion_xl_instantid_img2img import StableDiffusionXLInstantIDImg2ImgPipeline

controlnet = ControlNetModel.from_pretrained(
    "/content/instantid/ControlNetModel", torch_dtype=torch.float16,
)
pipe = StableDiffusionXLInstantIDImg2ImgPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    controlnet=controlnet,
    torch_dtype=torch.float16,
).to("cuda")
pipe.load_ip_adapter_instantid("/content/instantid/ip-adapter.bin")
print("img2img 파이프라인 로딩 완료")

img2img는 원본 이미지를 입력으로 받아 `strength`만큼 변형한다.
strength가 낮으면 원본이 많이 남고, 높으면 새로 그린 것에 가까워진다.
0.3 / 0.5 / 0.7 세 값을 비교해 원본 보존과 얼굴 교체가 양립하는 구간을 찾는다.

In [ ]:
emb, kps_img, src_pil = prep_instantid(
    "/content/ref_woman.png",
    f"{TEST_NORMAL}/normal_01_short_dark.jpg",
)

w, h = src_pil.size
scale = 1024 / max(w, h)
nw, nh = (int(w * scale) // 8) * 8, (int(h * scale) // 8) * 8
src_r, kps_r = src_pil.resize((nw, nh)), kps_img.resize((nw, nh))

outs = {}
for s in [0.3, 0.5, 0.7]:
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    out = pipe(
        prompt="a person in a hair salon, natural lighting, photorealistic",
        negative_prompt="blurry, low quality, deformed, watermark, text",
        image_embeds=emb,
        image=src_r,                    # 원본 이미지 (img2img 입력)
        control_image=kps_r,            # 얼굴 키포인트 (ControlNet 조건)
        strength=s,
        controlnet_conditioning_scale=0.8,
        ip_adapter_scale=0.8,
        num_inference_steps=30,
        guidance_scale=5.0,
        generator=torch.Generator("cuda").manual_seed(42),
    ).images[0]
    outs[s] = out
    print(f"strength {s}: {round(time.time()-t0,1)}초 / "
          f"{round(torch.cuda.max_memory_allocated()/1024**3,2)}GB")

fig, axes = plt.subplots(1, 4, figsize=(20, 6))
axes[0].imshow(src_r); axes[0].set_title("original"); axes[0].axis("off")
for ax, (s, o) in zip(axes[1:], outs.items()):
    ax.imshow(o); ax.set_title(f"strength {s}"); ax.axis("off")
plt.tight_layout(); plt.show()

### 조합 2 전체 생성

strength 0.4로 고정. 0.3~0.5 구간에서 원본 구조가 유지되며,
0.7은 헤어·배경이 변형되어 상한선을 벗어난다.

참조 얼굴은 성별에 맞춰 배정한다.

In [ ]:
import os, json

OUT_DIR2 = f"{BASE}/outputs/combo2"
os.makedirs(OUT_DIR2, exist_ok=True)

SEEDS = [42, 123, 777]
records2 = []

for f in list_images(TEST_NORMAL):
    ref = "/content/ref_man.png" if "male" in f else "/content/ref_woman.png"
    emb, kps_img, src_pil = prep_instantid(ref, f"{TEST_NORMAL}/{f}")

    # 조합 5와 동일 조건: 긴 변 1024, 8의 배수
    w, h = src_pil.size
    scale = 1024 / max(w, h)
    nw, nh = (int(w * scale) // 8) * 8, (int(h * scale) // 8) * 8
    src_r, kps_r = src_pil.resize((nw, nh)), kps_img.resize((nw, nh))

    for seed in SEEDS:
        torch.cuda.reset_peak_memory_stats()
        t0 = time.time()
        out = pipe(
            prompt="a person in a hair salon, natural lighting, photorealistic",
            negative_prompt="blurry, low quality, deformed, watermark, text",
            image_embeds=emb,
            image=src_r,
            control_image=kps_r,
            strength=0.4,
            controlnet_conditioning_scale=0.8,
            ip_adapter_scale=0.8,
            num_inference_steps=30,
            guidance_scale=5.0,
            generator=torch.Generator("cuda").manual_seed(seed),
        ).images[0]

        name = f"{f.replace('.jpg','')}_seed{seed}.png"
        out.save(f"{OUT_DIR2}/{name}")
        records2.append({
            "src": f, "output": name, "ref": os.path.basename(ref),
            "seed": seed, "strength": 0.4, "size": [nw, nh],
            "gen_sec": round(time.time() - t0, 1),
            "vram_gb": round(torch.cuda.max_memory_allocated() / 1024**3, 2),
        })
        print(f"{name}  {records2[-1]['gen_sec']}초  {records2[-1]['vram_gb']}GB")

with open(f"{OUT_DIR2}/meta.json", "w") as fp:
    json.dump(records2, fp, indent=2, ensure_ascii=False)

print(f"\n총 {len(records2)}장 → {OUT_DIR2}")

### 조합 2 결과 확인

헤어·의상·배경이 유지되는지, 5장 전부에서 안정적인지 확인한다.
특히 얼굴이 작은 normal_03과 각도가 있는 normal_05를 본다.

In [ ]:
from PIL import Image

files2 = sorted(f for f in os.listdir(OUT_DIR2) if f.endswith(".png"))

fig, axes = plt.subplots(5, 3, figsize=(12, 20))
for ax, f in zip(axes.flat, files2):
    ax.imshow(Image.open(f"{OUT_DIR2}/{f}"))
    ax.set_title(f.replace(".png", "").replace("normal_", ""), fontsize=8)
    ax.axis("off")
plt.tight_layout(); plt.show()

### 조합 2 관찰 정리

**설정**: InstantID img2img / SDXL base / strength 0.4 / steps 30 / guidance 5.0 /
controlnet_conditioning_scale 0.8 / ip_adapter_scale 0.8

**실측**

| 항목 | 값 |
|---|---|
| 로딩 | 49.4초 (다운로드 14.2GB) |
| 생성 | 6.8~10.9초 (평균 7.6초) |
| VRAM | 12.46~13.66GB |

**성립 조건**

기본 text2img 파이프라인은 원본 픽셀을 입력받지 않아 헤어·의상·배경이 전부
재생성된다. img2img 파이프라인이 필수다.

strength 0.3~0.5에서 원본 구조가 유지되고 0.7부터 헤어·배경이 변형된다.
조합 5와 달리 strength가 실제 변형 강도로 동작해 **원본 유사도 조절이 가능하다.**

**케이스별 결과 (strength 0.4)**

| 파일 | 결과 |
|---|---|
| normal_01 | 헤어·배경·의상 유지. 성공 |
| normal_02 | 원본과 인상이 거의 같아 얼굴 교체 효과가 약함. 전체적으로 뭉개짐 |
| normal_03 | **인종 변경**(백인→동양인), **배경 완전 변형** |
| normal_04 | **인종 변경**(백인→동양인), 헤어 변형 |
| normal_05 | **헤어 변형**(앞가르마→슬릭백), 배경 흐려짐 |

**문제점**

**참조 얼굴이 인종을 강제한다.** 조합 5는 CLIP으로 프롬프트를 자동 생성해 대응했으나,
조합 2는 참조 이미지가 인종을 결정하므로 인종별 참조 풀이 필요하다. 운영 복잡도가 증가한다.

**같은 strength에서도 사진마다 보존 정도가 다르다.** normal_01은 유지되고
03·04·05는 변형되었다. 안정성이 낮다.

**전체 화질이 저하된다.** 이미지 전체를 재생성하므로 원본 디테일이

# 조합 3 — InstantID + RealVisXL

조합 2와 방식은 동일하고 베이스 모델만 RealVisXL V5.0으로 교체한다.
베이스 모델이 결과에 미치는 영향을 분리 측정하는 것이 목적이다.

조합 2와 동일 조건: img2img / strength 0.4 / steps 30 / guidance 5.0 /
동일 참조 얼굴 / 동일 seed

In [ ]:
# 조합 2 파이프라인 해제 후 베이스만 RealVisXL로 교체
del pipe
print("해제 후:", release(), "GB")

controlnet = ControlNetModel.from_pretrained(
    "/content/instantid/ControlNetModel", torch_dtype=torch.float16,
)
pipe = StableDiffusionXLInstantIDImg2ImgPipeline.from_pretrained(
    "SG161222/RealVisXL_V5.0",          # ← 베이스만 변경
    controlnet=controlnet,
    torch_dtype=torch.float16,
).to("cuda")
pipe.load_ip_adapter_instantid("/content/instantid/ip-adapter.bin")
print("RealVisXL 파이프라인 로딩 완료")

In [ ]:
OUT_DIR3 = f"{BASE}/outputs/combo3"
os.makedirs(OUT_DIR3, exist_ok=True)

records3 = []

for f in list_images(TEST_NORMAL):
    ref = "/content/ref_man.png" if "male" in f else "/content/ref_woman.png"
    emb, kps_img, src_pil = prep_instantid(ref, f"{TEST_NORMAL}/{f}")

    w, h = src_pil.size
    scale = 1024 / max(w, h)
    nw, nh = (int(w * scale) // 8) * 8, (int(h * scale) // 8) * 8
    src_r, kps_r = src_pil.resize((nw, nh)), kps_img.resize((nw, nh))

    for seed in SEEDS:
        torch.cuda.reset_peak_memory_stats()
        t0 = time.time()
        out = pipe(
            prompt="a person in a hair salon, natural lighting, photorealistic",
            negative_prompt="blurry, low quality, deformed, watermark, text",
            image_embeds=emb,
            image=src_r,
            control_image=kps_r,
            strength=0.4,
            controlnet_conditioning_scale=0.8,
            ip_adapter_scale=0.8,
            num_inference_steps=30,
            guidance_scale=5.0,
            generator=torch.Generator("cuda").manual_seed(seed),
        ).images[0]

        name = f"{f.replace('.jpg','')}_seed{seed}.png"
        out.save(f"{OUT_DIR3}/{name}")
        records3.append({
            "src": f, "output": name, "ref": os.path.basename(ref),
            "seed": seed, "strength": 0.4, "size": [nw, nh],
            "gen_sec": round(time.time() - t0, 1),
            "vram_gb": round(torch.cuda.max_memory_allocated() / 1024**3, 2),
        })
        print(f"{name}  {records3[-1]['gen_sec']}초  {records3[-1]['vram_gb']}GB")

with open(f"{OUT_DIR3}/meta.json", "w") as fp:
    json.dump(records3, fp, indent=2, ensure_ascii=False)

print(f"\n총 {len(records3)}장 → {OUT_DIR3}")

In [ ]:
files3 = sorted(f for f in os.listdir(OUT_DIR3) if f.endswith(".png"))

fig, axes = plt.subplots(5, 3, figsize=(12, 20))
for ax, f in zip(axes.flat, files3):
    ax.imshow(Image.open(f"{OUT_DIR3}/{f}"))
    ax.set_title(f.replace(".png", "").replace("normal_", ""), fontsize=8)
    ax.axis("off")
plt.tight_layout(); plt.show()

### 조합 2·3 비교 결론

**성능 수치는 동일하다.** 생성 시간 6.7~10.8초, VRAM 12.46~13.66GB로
조합 2와 차이가 없다. 베이스 모델 크기가 같아 연산량이 동일하기 때문이다.
로딩만 RealVisXL이 2배 이상 느리다(shard 분할).

**품질은 RealVisXL이 소폭 우세하다.**
- normal_03: 배경 구조가 더 보존되고 원본 금발 색이 유지됨 (조합 2는 갈색으로 변함)
- normal_01: 배경 디테일이 더 선명함

**핵심 문제는 동일하게 남는다.**
- normal_03·04: 인종 변경 (백인 → 동양인)
- normal_04·05: 헤어 변형

**결론**: 베이스 모델은 화질·색 재현에 영향을 주지만 원본 보존 성능은 바꾸지 못한다.
인종 변경과 헤어 변형은 방식에서 기인하며 베이스 교체로 해결되지 않는다.

따라서 조합 4에 대한 RealVisXL 추가 검증은 실익이 없다고 판단해 진행하지 않는다.
(계획서 3장 "조합 2·3에서 유의미한 차이가 확인되면 조합 4·5에도 추가 검증" 조항에 대한 판단)